# 07 · Distillation on Colab

Trains a student that sees only the 8 frames and the question (no measuring part, no calculator) and compares it with
the current system. Use a GPU runtime with 24 GB or more (**L4** or A100); a T4 is too small.

**How it survives a disconnect.** The workspace lives on the fast local disk (`/content/workspace`). After every step
the small files (weights, checkpoints, results, logs, videos) are packed into one archive on your Google Drive. After a
disconnect, run the cells again from the top: the archive is unpacked, finished steps are skipped, and only the image
feature cache (about 15 GPU minutes) is rebuilt. Set `USE_DRIVE = False` to skip Drive.

**Continuing from the desktop.** Put the archive made on the desktop (`spatial_llm_distill_backup.tar`, about 0.8 GB)
into the top folder of your Google Drive before running. Section 1 then only rebuilds the image-feature cache.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
USE_DRIVE = True          # keep weights, checkpoints and results on Google Drive
import os, subprocess
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
BACKUP = '/content/drive/MyDrive/spatial_llm_distill_backup.tar'

if not os.path.exists('/content/repo'):
    !git clone -q -b distillation https://github.com/NateHu203/spatial-llm-out-of-view.git /content/repo
%cd /content/repo
!git pull -q
!pip -q install -r requirements.txt
os.environ['SPATIAL_LLM_WORK'] = '/content/workspace'
os.environ['PYTHONPATH'] = '/content/repo/src'
os.makedirs('/content/workspace', exist_ok=True)


def restore():
    if USE_DRIVE and os.path.exists(BACKUP):
        subprocess.run(['tar', '-xf', BACKUP, '-C', '/content/workspace'], check=True)
        print('restored from', BACKUP)


def backup():
    """everything except the image-feature cache (18 GB, rebuilt in minutes) -> one archive on Drive"""
    if USE_DRIVE:
        subprocess.run(['tar', '--exclude=./sts_data/vcache', '-cf', BACKUP + '.tmp', '-C', '/content/workspace', '.'], check=True)
        os.replace(BACKUP + '.tmp', BACKUP)
        print('backed up to', BACKUP)


restore()

# while a run is going: copy its checkpoint, loss curve and tested scenes to Drive every 10 minutes,
# so a disconnect in the middle of a run loses at most 10 minutes
import glob, shutil, threading, time
LIVE = '/content/drive/MyDrive/spatial_llm_distill_live'
if USE_DRIVE:
    os.makedirs(LIVE, exist_ok=True)
    for f in glob.glob(LIVE + '/*'):
        if not f.endswith('.tmp') and not os.path.exists('/content/workspace/' + os.path.basename(f)):
            shutil.copy(f, '/content/workspace'); print('continuing from', os.path.basename(f))

    def keep_live():
        while True:
            time.sleep(600)
            for pat in ('*.ckpt', '*.partial', '*_train.jsonl'):
                for f in glob.glob('/content/workspace/' + pat):
                    try:
                        shutil.copy(f, LIVE + '/' + os.path.basename(f) + '.tmp')
                        os.replace(LIVE + '/' + os.path.basename(f) + '.tmp', LIVE + '/' + os.path.basename(f))
                    except OSError as e:
                        print('live copy failed:', e)
    if not any(t.name == 'keep_live' for t in threading.enumerate()):
        threading.Thread(target=keep_live, name='keep_live', daemon=True).start()

## 1 · What the student needs first

Videos, the image-feature cache, model W (the starting LoRA), the measuring part's tracks and the tuned calculator
rules. On the authors' 16 GB desktop: data 19 min, cache 14 min, model W 8 min, estimator 3 min, rules 9 min. The data
and rules steps use the CPU only and are slower on Colab's fewer cores. Finished steps are skipped.

In [ ]:
%cd /content/repo
!python run.py data
backup()
!python run.py cache model_w estimator
backup()
!python run.py rules
backup()

## 2 · The experiments that are still open

`tools/distill_queue.py` skips every experiment whose result file is already in the workspace (the archive from the
desktop holds all finished ones) and runs the rest. Each one is backed up to Drive when it ends; after a disconnect,
run the notebook again from the top and training continues from the last backed-up state.

| Experiment | What it is | On the 16 GB desktop |
|---|---|---|
| `record_from_locate_s0` | the student writes every sighting frame by frame, then the state, then the answer; it starts from the weights that already localise (`d_locate_s0.pt`) | about 2 h |
| `locate_long_s0` | the localisation diagnostic with 9000 steps instead of 3000 | about 35 min |

In [ ]:
NAMES = ['record_from_locate_s0', 'locate_long_s0']
%cd /content/repo
for name in NAMES:
    !python tools/distill_queue.py run {name}
    backup()
!python tools/distill_queue.py status

## 3 · Any other arm (optional)

`--target` is `answer`, `state`, `motion`, `record` or `locate`; `--seed` picks the seed; `--resume` continues an
interrupted run. Tested so far (eval9, twin accuracy, chance is 25%): `answer` 22.6 (3 seeds), `state` 24.2 (3 seeds),
`motion` 12.5, `record` 17.4, `record` started from the `locate` weights 21.2. `locate` (position error on eval3): 0.278 m at 3000 steps (3 seeds), 0.151 m at 9000 steps (1 seed).

In [ ]:
ARMS = []                  # e.g. ['record'] ; nothing is run while this is empty
SEED = 1
%cd /content/workspace
for arm in ARMS:
    !python ../src/distill.py --target {arm} --seed {SEED} --save_lora d_{arm}_s{SEED}.pt --out res_d_{arm}_s{SEED}.jsonl --resume
    backup()

## 4 · Results

In [ ]:
import glob, json
for path in sorted(glob.glob('/content/workspace/res_d_*.jsonl')):
    if path.endswith('_train.jsonl'):
        continue
    for line in open(path):
        r = json.loads(line)
        print(os.path.basename(path), '|', r['arm'], '| tested on', r['split'], '| steps', r['n_train'])
        for k, v in r['summary'].items():
            print('   ', k, v)

In [ ]:
# optional: download the small result files (the written text of every question is inside the res_d_*.jsonl files)
!cd /content/workspace && zip -q -r /content/distill_results.zip . -i 'res_d_*' 'logs/*' 'state_rules_results.json'
from google.colab import files
files.download('/content/distill_results.zip')